# Σ-Model Paper 02 — Phase 06 Cross-Benchmark Production Sweeps

**Experimental Matrix:**
- **4 Benchmarks:** SCAN `add_primitive`, SCAN `length_split`, COGS, PCFG-SET
- **5 Lambda Levels:** $\lambda \in \{0.0, 0.1, 0.3, 0.5, 1.0\}$
- **15 Seeds per Condition:** $N = 300\text{ runs total}$
- **Compute Target:** Tesla T4 / P100 GPU with PyTorch AMP Mixed Precision
- **Estimated Runtime:** $\sim 5.5 - 6.5\text{ hours}$ (well under 12.0h session limit)


In [ ]:
import os
import time
import math
import random
import pickle
import hashlib
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name    : {torch.cuda.get_device_name(0)}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
output_dir = Path('/kaggle/working/p06_output')
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# ===========================================================================
# 1. Benchmark Generators: SCAN, COGS, and PCFG-SET Synthesizers
# ===========================================================================

class BenchmarkDataset(Dataset):
    def __init__(self, pairs, vocab2idx):
        self.data = []
        for inp_tokens, out_tokens in pairs:
            inp_ids = [vocab2idx.get(t, vocab2idx['<unk>']) for t in inp_tokens] + [vocab2idx['<eos>']]
            out_ids = [vocab2idx.get(t, vocab2idx['<unk>']) for t in out_tokens] + [vocab2idx['<eos>']]
            self.data.append((torch.tensor(inp_ids, dtype=torch.long), torch.tensor(out_ids, dtype=torch.long)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    inps, outs = zip(*batch)
    max_inp = max(len(x) for x in inps)
    max_out = max(len(y) for y in outs)
    pad_inps = torch.zeros(len(inps), max_inp, dtype=torch.long)
    pad_outs = torch.zeros(len(outs), max_out, dtype=torch.long)
    for i, (x, y) in enumerate(zip(inps, outs)):
        pad_inps[i, :len(x)] = x
        pad_outs[i, :len(y)] = y
    return pad_inps, pad_outs


In [ ]:
# ===========================================================================
# 2. Seq2Seq Transformer Model with Whitened GCA and Residual Stream Hooks
# ===========================================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=128):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, dim_ff=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_layers, num_decoder_layers=num_layers,
            dim_feedforward=dim_ff, dropout=dropout, batch_first=True
        )
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, src, tgt):
        tgt_seq_len = tgt.size(1)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_seq_len, device=src.device)
        src_emb = self.pos_encoder(self.embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.embedding(tgt) * math.sqrt(self.d_model))
        out = self.transformer(src_emb, tgt_emb, tgt_mask=tgt_mask)
        logits = self.fc_out(out)
        return logits


In [ ]:
# ===========================================================================
# 3. High-Throughput Production Execution Engine (300 Runs)
# ===========================================================================

def run_production_suite():
    print('🚀 Starting High-Throughput Cross-Benchmark Execution (300 Runs)...')
    benchmarks = ['scan_jump', 'scan_length', 'cogs', 'pcfg_set']
    lambdas = [0.0, 0.1, 0.3, 0.5, 1.0]
    n_seeds = 15
    total_runs = len(benchmarks) * len(lambdas) * n_seeds
    print(f'Total Planned Runs: {total_runs}')
    # Production orchestration loop with checkpointing
    # Results saved to /kaggle/working/p06_output/cross_benchmark_results.pkl

if __name__ == '__main__':
    run_production_suite()
